# Step 1b — Baseline GPT-2 Fine-Tuning (ε = ∞, no DP)

Fine-tune GPT-2 small on the tokenized dataset prepared by `step1a_data.ipynb`.
This is the `ε = ∞` baseline run — no differential privacy.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/CSC2412/dp-finetuned-llms'
print('Drive mounted. Project dir:', PROJECT_DIR)

## 2. Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate

## 3. Imports & Config

In [ ]:
import os
import math
import json
import torch
import numpy as np
from datasets import load_from_disk
from transformers import (
    GPT2TokenizerFast,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# ── Config ─────────────────────────────────────────────────────────────────────
SEED           = 42
BATCH_SIZE     = 8
EPOCHS         = 3
LR             = 5e-5
CHECKPOINT_DIR = f'{PROJECT_DIR}/checkpoints/baseline'

torch.manual_seed(SEED)
np.random.seed(SEED)

## 4. Load Tokenized Data from Drive

In [ ]:
train_ds  = load_from_disk(f'{PROJECT_DIR}/data/train_ds')
val_ds    = load_from_disk(f'{PROJECT_DIR}/data/val_ds')
tokenizer = GPT2TokenizerFast.from_pretrained(f'{PROJECT_DIR}/data/tokenizer')

train_ds.set_format('torch')
val_ds.set_format('torch')

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

## 5. Load GPT-2 Small

In [ ]:
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.resize_token_embeddings(len(tokenizer))
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'GPT-2 small parameters: {n_params/1e6:.1f}M')

## 6. Fine-Tune (no DP)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # causal LM, not masked LM
)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=100,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=50,
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

trainer.train()

## 7. Evaluate — Perplexity on Validation Set

In [ ]:
eval_results = trainer.evaluate()

log_perplexity = eval_results['eval_loss']  # cross-entropy loss = log-perplexity (base e)
perplexity     = math.exp(log_perplexity)

print(f'\n=== Baseline Results (ε = ∞, no DP) ===')
print(f'Log-perplexity (ln):  {log_perplexity:.4f}')
print(f'Perplexity:           {perplexity:.2f}')

results = {
    'epsilon': float('inf'),
    'log_perplexity': log_perplexity,
    'perplexity': perplexity,
    'epochs': EPOCHS,
}

results_path = f'{PROJECT_DIR}/results/baseline_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {results_path}')

## 8. Save Model to Drive

In [ ]:
model_save_path = f'{PROJECT_DIR}/checkpoints/baseline/final'
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f'Model saved to {model_save_path}')

## 9. Sanity Check — Generate Text

In [ ]:
model.eval()
prompt = "I have been feeling really anxious lately"
inputs = tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )

print('Prompt:', prompt)
print('Generated:', tokenizer.decode(output[0], skip_special_tokens=True))

---
**Next:** Run `step2_canaries.ipynb` to insert canaries and implement exposure scoring.